## Figure 1a — accepted event-context carrier

Plot action: verify and package the user-confirmed Figure 1a source image as
PNG and PDF without changing its pixels or scientific content.

Inputs: `figures/source/figure01a_source.png`, the accepted Oct–Sep
MERRA-2 2019/2020 and WACCM year-0008 ozone-evolution image. The SHA-256 gate
prevents a different image from being substituted silently. The exact legacy
raw-to-envelope calculation has not yet been recovered, so this block is an
explicit carrier, not a claimed scientific reconstruction.

Outputs: `figure01a_o3_event_context.png` and PDF.


In [ ]:
from __future__ import annotations

import hashlib
import os
import shutil
from pathlib import Path

from PIL import Image

EXPECTED_SOURCE_SHA256 = "f5daf497b72eee444c6d8435441a9f346970a9932efca02afe1bdab6b6b51f02"


def discover_repository_root() -> Path:
    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "figures" / "source").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError("Cannot locate the Paper 1 code directory")


REPOSITORY_ROOT = discover_repository_root()
DERIVED_ROOT = Path(
    os.environ.get("PAPER1_DERIVED_ROOT", str(REPOSITORY_ROOT / "work"))
).expanduser().resolve()
if (
    DERIVED_ROOT == Path(DERIVED_ROOT.anchor)
    or DERIVED_ROOT == REPOSITORY_ROOT
    or DERIVED_ROOT in REPOSITORY_ROOT.parents
):
    raise PermissionError("PAPER1_DERIVED_ROOT is not a safe output directory")
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if OUTPUT_DIR != DERIVED_ROOT and DERIVED_ROOT not in OUTPUT_DIR.parents:
    raise PermissionError("PAPER1_FIGURE_ROOT must stay inside PAPER1_DERIVED_ROOT")


def canonical_path(relative: str) -> Path:
    path = REPOSITORY_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def save_figure(source_png: Path, stem: str) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Unsafe figure stem: {stem!r}")
    digest = hashlib.sha256(source_png.read_bytes()).hexdigest()
    if digest != EXPECTED_SOURCE_SHA256:
        raise ValueError(f"Figure 1a source hash mismatch: {digest}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        shutil.copyfile(source_png, temporary_png)
        with Image.open(source_png) as opened:
            opened.convert("RGB").save(temporary_pdf, "PDF", resolution=300.0)
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved accepted Figure 1a carrier: {png}")
    print(f"saved accepted Figure 1a carrier: {pdf}")


source_png = REPOSITORY_ROOT / "figures" / "source" / "figure01a_source.png"
if not source_png.is_file():
    raise FileNotFoundError(f"Missing accepted Figure 1a source: {source_png}")
save_figure(source_png, "figure01a_o3_event_context")


## Appendix B — unsmoothed daily event ozone profiles

Input: canonical `ozone/event_profile_bootstrap5000.nc`. It contains daily MERRA-2 2020 and WACCM year-0008 60–90° N ozone anomalies, target-excluded daily climatologies, and a 5,000-resample 95% bootstrap significance mask. The cell requires `temporal_smoothing_days=0`; it does not smooth the anomaly, climatology, or significance mask. Black contours show the corresponding target-excluded climatology. Hatching marks grid cells that are not significant.

Output: `figure01bc_waccm0008_merra2_2020_o3_anomaly_1to100hpa.png` and PDF, with WACCM at left, MERRA-2 at right, and publication-scale typography.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "work"
DATA_ROOT = Path(os.environ.get("PAPER1_ARCHIVE_ROOT", str(REPOSITORY_ROOT / "data"))).expanduser().resolve()
INPUT_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        DATA_ROOT,
        DATA_ROOT / "B2000WCN001002_timefixed",
        DATA_ROOT / "BWCN",
        DATA_ROOT / "Hindcast",
        DATA_ROOT / "WACCM" / "march_hindcast",
        DATA_ROOT / "MERRA2M2I6NPANA",
        DATA_ROOT / "MERRA2_Processed",
        DATA_ROOT / "MLS",
        DATA_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == DATA_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if root == Path(root.anchor):
        raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
    for protected in INPUT_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored work tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run the analysis notebooks or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    import sys as _sys
    style_directory = str(REPOSITORY_ROOT / "figures")
    if style_directory not in _sys.path:
        _sys.path.insert(0, style_directory)
    from paper_style import apply_paper_style
    apply_paper_style(figure, stem)
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
product = load_dataset(
    "ozone/event_profile_bootstrap5000.nc",
    (
        "merra2_o3_anomaly", "merra2_o3_climatology",
        "merra2_ci_low", "merra2_ci_high", "merra2_significant",
        "waccm_o3_anomaly", "waccm_o3_climatology",
        "waccm_ci_low", "waccm_ci_high", "waccm_significant",
    ),
)
if int(product.attrs.get("bootstrap_replicates", -1)) != 5000:
    raise ValueError("Event-profile product is not bootstrap-5000")
if product.attrs.get("target_excluded") not in (True, "True", "true", 1):
    raise ValueError("Bootstrap baseline must exclude the target event")
if str(product.attrs.get("percentile_bounds")) != "2.5,97.5":
    raise ValueError("Expected the canonical 2.5/97.5 percentile interval")
if int(product.attrs.get("temporal_smoothing_days", -1)) != 0:
    raise ValueError("Appendix B requires unsmoothed daily profile anomalies")
if str(product.attrs.get("profile_anomaly_temporal_resolution", "")).lower() != "daily":
    raise ValueError("Appendix B requires daily profile-anomaly metadata")

from matplotlib.patches import Patch

season_day = np.asarray(product["season_day"].values, dtype=int)
keep = (season_day >= 92) & (season_day <= 242)
x = season_day[keep] - 92
levels = np.linspace(-1.4, 1.4, 15)
climatology_levels = np.arange(0.4, 6.9, 0.4)
figure, axes = plt.subplots(
    1, 2, figsize=(15.2, 6.0), sharex=True, sharey=True,
    constrained_layout=True,
)
panels = (
    (
        axes[0], "waccm_o3_anomaly", "waccm_o3_climatology",
        "waccm_significant", "waccm_pressure_hpa", "(a) WACCM year 0008",
    ),
    (
        axes[1], "merra2_o3_anomaly", "merra2_o3_climatology",
        "merra2_significant", "merra2_pressure_hpa", "(b) MERRA-2 2020",
    ),
)
mappable = None
for axis, anomaly_name, climatology_name, significant_name, pressure_name, title in panels:
    pressure = np.asarray(product[pressure_name].values, dtype=float)
    anomaly = np.asarray(
        product[anomaly_name].isel(season_day=np.where(keep)[0]).values,
        dtype=float,
    ).T
    climatology = np.asarray(
        product[climatology_name].isel(season_day=np.where(keep)[0]).values,
        dtype=float,
    ).T
    significant = np.asarray(
        product[significant_name].isel(season_day=np.where(keep)[0]).values,
        dtype=bool,
    ).T
    mappable = axis.contourf(
        x, pressure, anomaly, levels=levels, cmap="RdBu_r", extend="both",
    )
    contours = axis.contour(
        x, pressure, climatology, levels=climatology_levels,
        colors="black", linewidths=0.85,
    )
    axis.clabel(contours, fontsize=7.0)
    nonsignificant = np.where(np.isfinite(anomaly), ~significant, np.nan)
    axis.contourf(
        x, pressure, nonsignificant.astype(float),
        levels=[0.5, 1.5], colors="none", hatches=["///"],
    )
    axis.set_yscale("log")
    axis.invert_yaxis()
    axis.set_ylim(100.0, 1.0)
    axis.set_yticks([1, 3, 10, 30, 100])
    axis.set_yticklabels(["1", "3", "10", "30", "100"])
    axis.set_xlim(0, 150)
    axis.set_xticks([0, 31, 59, 90, 120], ["Jan", "Feb", "Mar", "Apr", "May"])
    axis.set_title(title, fontweight="bold")
    axis.set_xlabel("Calendar month")
    axis.grid(axis="x", color="0.87", linewidth=0.6)
axes[0].set_ylabel("Pressure (hPa)")
legend_patch = Patch(
    facecolor="white", edgecolor="black", hatch="///",
    label="Not significant (p >= 0.05)",
)
axes[1].legend(handles=[legend_patch], loc="upper right", frameon=False)
colorbar = figure.colorbar(mappable, ax=axes, pad=0.025, aspect=32)
colorbar.set_label("O$_3$ anomaly (ppmv)")
colorbar.set_ticks(np.arange(-1.2, 1.21, 0.4))
figure.suptitle("Polar-cap daily O$_3$ anomalies in WACCM year 0008 and MERRA-2 2020")
save_figure(
    figure,
    "figure01bc_waccm0008_merra2_2020_o3_anomaly_1to100hpa",
)
